In [ ]:
import pandas as pd
import numpy as np

from collections import deque
from math import e, factorial

DATASET_PATH = "data/compilado.csv"

FEATURE_COLUMN_INDEX = 1

WINDOW_START = 57
WINDOW_END = 61

ERROR_THRESHOLD = 1.85
ATTACK_INCREMENT_THRESHOLD = 12

In [ ]:
dataframe = pd.read_csv(DATASET_PATH)

traffic_series = dataframe.iloc[:, FEATURE_COLUMN_INDEX].tolist()

print(f"Dataset loaded with {len(traffic_series)} records.")

26.08427732020962


In [ ]:
def calculate_pma_weights(window_size):
    weights = []

    for i in range(1, window_size + 1):
        poisson_weight = ((window_size ** i) * (e ** (-window_size))) / factorial(i)
        weights.append(poisson_weight * 2)

    return weights


def pma(values):
    weights = calculate_pma_weights(len(values))
    return sum(value * weight for value, weight in zip(values, weights))

In [ ]:
def relative_amplitude(values):
    mean_value = sum(values) / len(values)

    if mean_value == 0:
        return 0

    return (max(values) - min(values)) / mean_value


def calculate_power_measure(values):
    power = relative_amplitude(values)
    n = len(values)

    return [(i / n) ** power for i in range(n, 0, -1)]


def choquet_integral(values):
    sorted_values = sorted(values)
    total_sum = sum(values)

    if total_sum == 0:
        return 0

    normalized_values = [value / total_sum for value in sorted_values]
    weights = calculate_power_measure(values)

    result = 0

    for i in range(len(values)):
        if i == 0:
            result += normalized_values[i] * weights[i]
        else:
            result += abs(normalized_values[i] - normalized_values[i - 1]) * weights[i]

    return result * total_sum

In [ ]:
def detect_attention(current_value, predicted_value, error_threshold=ERROR_THRESHOLD):
    if predicted_value > error_threshold * current_value:
        return 1

    if current_value > error_threshold * predicted_value:
        return 1

    return 0

In [ ]:
def calculate_metrics(tp, tn, fp, fn):
    total = tp + tn + fp + fn

    accuracy = (tp + tn) / total if total != 0 else 0
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    f1_score = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) != 0
        else 0
    )

    return accuracy, precision, recall, specificity, f1_score

In [ ]:
def process_window(window_size, traffic_series):
    sliding_window = deque(traffic_series[:window_size], maxlen=window_size)
    prediction_values = traffic_series[window_size:]

    counters = {
        "real": {"attention": 0, "attack": 0},
        "choquet": {"attention": 0, "attack": 0},
    }

    increments = {
        "real": 0,
        "choquet": 0,
    }

    confusion_matrix = {
        "TP": 0,
        "FP": 0,
        "FN": 0,
        "TN": 0,
    }

    previous_value = 0

    for current_observed_value in prediction_values:
        penultimate_value = previous_value
        previous_value = sliding_window[-1]

        if penultimate_value != 0 and previous_value != 0:
            previous_attention = counters["real"]["attention"]

            counters["real"]["attention"] += detect_attention(
                penultimate_value,
                previous_value
            )

            if counters["real"]["attention"] > previous_attention:
                increments["real"] += 1
            else:
                increments["real"] = 0

            real_attack = int(increments["real"] >= ATTACK_INCREMENT_THRESHOLD)

            if real_attack:
                counters["real"]["attack"] += 1
        else:
            real_attack = 0

        predicted_value = round(choquet_integral(list(sliding_window)))

        previous_attention = counters["choquet"]["attention"]

        counters["choquet"]["attention"] += detect_attention(
            previous_value,
            predicted_value
        )

        if counters["choquet"]["attention"] > previous_attention:
            increments["choquet"] += 1
        else:
            increments["choquet"] = 0

        choquet_attack = int(increments["choquet"] >= ATTACK_INCREMENT_THRESHOLD)

        if choquet_attack:
            counters["choquet"]["attack"] += 1

        if real_attack == 1 and choquet_attack == 1:
            confusion_matrix["TP"] += 1
        elif real_attack == 1 and choquet_attack == 0:
            confusion_matrix["FP"] += 1
        elif real_attack == 0 and choquet_attack == 1:
            confusion_matrix["FN"] += 1
        else:
            confusion_matrix["TN"] += 1

        sliding_window.append(current_observed_value)

    return counters, confusion_matrix

In [ ]:
results = []
confusion_results = []

for window_size in range(WINDOW_START, WINDOW_END + 1):
    counters, confusion_matrix = process_window(window_size, traffic_series)

    tp = confusion_matrix["TP"]
    fp = confusion_matrix["FP"]
    fn = confusion_matrix["FN"]
    tn = confusion_matrix["TN"]

    accuracy, precision, recall, specificity, f1_score = calculate_metrics(
        tp, tn, fp, fn
    )

    results.append({
        "window": window_size,
        "real_attacks": counters["real"]["attack"],
        "real_attention": counters["real"]["attention"],
        "choquet_attacks": counters["choquet"]["attack"],
        "choquet_attention": counters["choquet"]["attention"],
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1_score": f1_score,
    })

    confusion_results.append({
        "window": window_size,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    })

results_df = pd.DataFrame(results)
confusion_df = pd.DataFrame(confusion_results)


Janela | Real (Atk/Att) | CQ2 (Atk/Att)
    57 | 317/14532         | 3497/18322
    58 | 317/14531         | 3503/18337
    59 | 317/14530         | 3589/18447
    60 | 317/14529         | 3616/18479
    61 | 317/14528         | 3658/18465


In [1]:
import os

os.makedirs("results", exist_ok=True)

results_df.to_csv("results/choquet_metrics.csv", index=False)
confusion_df.to_csv("results/choquet_confusion_matrix.csv", index=False)

NameError: name 'results_df' is not defined